# Check object matching results

Examines per-catalog coverage, SNR distributions, and spatial layouts of the multi-catalog object match.

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

In [ ]:
tract = 8006

# Input catalog files and IDs for the object meta-match
all_obj_stats = tables_io.read(f'{tract}/all_match_object_stats.pq')
all_obj_assoc = tables_io.read(f'{tract}/all_match_object_assoc.pq')
object_in = tables_io.read(f'{tract}/object_reduced_{tract}.parq')
anacal_in = tables_io.read(f'{tract}/anacal_reduced_{tract}.parq')
desc_md_in = tables_io.read(f'{tract}/desc_md_match_object_joined.pq')
rubin_md_in = tables_io.read(f'{tract}/rubin_md_match_object_joined.pq')

all_files = [
    f'{tract}/object_reduced_{tract}.parq',
    f'{tract}/anacal_reduced_{tract}.parq',
    f'{tract}/desc_md_match_object_joined.pq',
    f'{tract}/rubin_md_match_object_joined.pq',
]
all_ids = list(range(len(all_files)))

In [ ]:
# Build wide joined table
joined_obj = hpmcm.output_tables.buildJoinedObjectTable(
    all_obj_stats,
    all_obj_assoc,
    all_files,
    all_ids,
)

## signal to noise distributions for various catalogs

In [ ]:
# SNR distributions split by which catalogs detected each object
bins = np.logspace(0, 5, 51)
cats = ['object', 'anacal', 'rubin', 'desc']
for i, cat in enumerate(cats):
    mask_bit = 1 << i
    mask = np.bitwise_and(joined_obj.catalog_mask, mask_bit) == mask_bit
    _ = plt.hist(joined_obj[mask].snr, bins=bins, alpha=0.1, label=cat)
_ = plt.legend()
_ = plt.xlabel('snr')
_ = plt.ylabel('object / 0.1')
_ = plt.xscale('log')
_ = plt.yscale('log')

#### SNR versus catalog mask

In [ ]:
# 2D histogram: which catalogs detected each object vs SNR
_ = plt.hist2d(joined_obj.catalog_mask, joined_obj.snr, bins=(np.linspace(0.5, 15.5, 16), np.logspace(0, 5, 51)), norm='log')
_ = plt.yscale('log')
_ = plt.xlabel('catalog mask')
_ = plt.ylabel('snr')

In [ ]:
_ = plt.hist(joined_obj.catalog_mask, bins=np.linspace(0.5, 15.5, 16))
_ = plt.yscale('log')
_ = plt.xlabel('catalog mask')

#### Objects were the matching failed to converge

In [ ]:
cut = np.bitwise_and(joined_obj.n_sib_objects==1, joined_obj.n_sib_srcs>4)
print(cut.sum())

#### Define masks and subsamples for good and bad matches

In [ ]:
good_mask = joined_obj.catalog_mask == 15
bad_mask = joined_obj.catalog_mask != 15
good_sub = joined_obj[good_mask]
bad_sub = joined_obj[bad_mask]

#### Plot good and bad matches across the tract

In [ ]:
_ = plt.scatter(bad_sub.ra[::5], bad_sub.dec[::5], s=1)
_ = plt.scatter(good_sub.ra[::5], good_sub.dec[::5], s=1)
_ = plt.xlabel('RA')
_ = plt.ylabel('DEC')

#### Plot good and bad matches across the cell

In [ ]:
_ = plt.scatter(bad_sub.x_cent[::10], bad_sub.y_cent[::10], s=1)
_ = plt.scatter(good_sub.x_cent[::10], good_sub.y_cent[::10], s=1)
_ = plt.xlabel(r'$x_{\rm cell}$ [pix]')
_ = plt.ylabel(r'$y_{\rm cell}$ [pix]')

In [ ]:
_ = plt.hist(bad_sub.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(good_sub.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.xlabel(r'$x_{\rm cent}$ [pix]')
_ = plt.ylabel('objects / pix')

#### Plot difference between centroid and object catalog position

In [ ]:
_ = plt.hist2d(3600*(good_sub.ra-good_sub.ra_0), 3600*(good_sub.dec-good_sub.dec_0),
               bins=(np.linspace(-0.8, 0.8, 81), np.linspace(-0.8, 0.8, 81)), norm='log')
_ = plt.xlabel(r'$x_{\rm cent} - x_{\rm obj}$ ["]')
_ = plt.ylabel(r'$y_{\rm cent} - y_{\rm obj}$ ["]')

#### Plot difference between centroid and anacal catalog position

In [ ]:
_ = plt.hist2d(3600*(good_sub.ra-good_sub.ra_1), 3600*(good_sub.dec-good_sub.dec_1),
               bins=(np.linspace(-0.8, 0.8, 81), np.linspace(-0.8, 0.8, 81)), norm='log')
_ = plt.xlabel(r'$x_{\rm cent} - x_{\rm anacal}$ ["]')
_ = plt.ylabel(r'$y_{\rm cent} - y_{\rm anacal}$ ["]')

## Color comparisons

#### Object colors v. anacal colors

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.r_gaapPsfFlux_0/good_sub.i_gaapPsfFlux_0) - 
        np.log10(good_sub.lsst_r_flux_gauss2_1/good_sub.lsst_i_flux_gauss2_1)
    ),
    bins=np.linspace(-1, 1, 101),
)
#_ = plt.yscale('log')
_ = plt.xlabel(r'$r-i_{\rm obj} - r-i_{\rm anacal}$ [mag]')
_ = plt.ylabel('objects / 0.02')
#_ = plt.xscale('log')

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.i_gaapPsfFlux_0/good_sub.z_gaapPsfFlux_0) - 
        np.log10(good_sub.lsst_i_flux_gauss2_1/good_sub.lsst_z_flux_gauss2_1)
    ),
    bins=np.linspace(-1, 1, 101),
)
_ = plt.yscale('log')
_ = plt.xlabel(r'$i-z_{\rm obj} - i-z_{\rm anacal}$ [mag]')
_ = plt.ylabel('objects / 0.02')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

In [ ]:
_ = plt.scatter(
    -2.5*(
        np.log10(good_sub.i_gaapPsfFlux_0/good_sub.z_gaapPsfFlux_0) - 
        np.log10(good_sub.lsst_i_flux_gauss2_1/good_sub.lsst_z_flux_gauss2_1)
    ),
    31.4-2.5*np.log10(good_sub.i_gaapPsfFlux_0),
    s=1,
)
#_ = plt.yscale('log')
_ = plt.xlabel(r'$(i-z)_{\rm obj} - (i-z)_{\rm anacal}$ [mag]')
_ = plt.ylabel('i [mag]')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

### object colors v. DESC MD colors

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.r_gaapPsfFlux_0/good_sub.i_gaapPsfFlux_0) - 
        np.log10(good_sub.flux_r_mean_2/good_sub.flux_i_mean_2)
    ),
    bins=np.linspace(-1, 1, 101),
)
_ = plt.yscale('log')
_ = plt.xlabel(r'$r-i_{\rm obj} - r-i_{\rm DESC MD}$ [mag]')
_ = plt.ylabel('objects / 0.02')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.i_gaapPsfFlux_0/good_sub.z_gaapPsfFlux_0) - 
        np.log10(good_sub.flux_i_mean_2/good_sub.flux_z_mean_2)
    ),
    bins=np.linspace(-1, 1, 101),
)
_ = plt.yscale('log')
_ = plt.xlabel(r'$i-z_{\rm obj} - i-z_{\rm DESC MD}$ [mag]')
_ = plt.ylabel('objects / 0.02')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

### object colors v. Rubin MD colors

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.r_gaapPsfFlux_0/good_sub.i_gaapPsfFlux_0) - 
        np.log10(good_sub.r_gaussFlux_mean_3/good_sub.i_gaussFlux_mean_3)
    ),
    bins=np.linspace(-1, 1, 101),
)
_ = plt.yscale('log')
_ = plt.xlabel(r'$r-i_{\rm obj} - r-i_{\rm Rubin MD}$ [mag]')
_ = plt.ylabel('objects / 0.02')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

In [ ]:
_ = plt.hist(
    -2.5*(
        np.log10(good_sub.i_gaapPsfFlux_0/good_sub.z_gaapPsfFlux_0) - 
        np.log10(good_sub.i_gaussFlux_mean_3/good_sub.z_gaussFlux_mean_3)
    ),
    bins=np.linspace(-1, 1, 101),
)
_ = plt.yscale('log')
_ = plt.xlabel(r'$i-z_{\rm obj} - i-z_{\rm Rubin MD}$ [mag]')
_ = plt.ylabel('objects / 0.02')

#_ = plt.yscale('log')
#_ = plt.xscale('log')

### Now look at g-band fluxes for object matches

In [ ]:
_ = plt.hist(-2.5*np.log10(good_sub.g_gaapPsfFlux_0) + 34.1, bins=np.linspace(20, 30, 101))
_ = plt.xlabel(r'$g$ [mag]')
_ = plt.ylabel('objects / 0.1 mag')

In [ ]:
_ = plt.hist(good_sub.g_gaapPsfFlux_0/good_sub.g_gaapPsfFluxErr_0, bins=np.logspace(0, 3, 101))
_ = plt.xscale('log')
_ = plt.yscale('log')
_ = plt.xlabel(r'$g / \delta g$')


In [ ]:
_ = plt.hist2d(
    good_sub.snr_3,
    good_sub.g_gaapPsfFlux_0/good_sub.g_gaapPsfFluxErr_0, 
    bins=(np.logspace(0, 4, 101), np.logspace(-1, 3, 101)),
    norm='log',
)
_ = plt.xscale('log')
_ = plt.yscale('log')
_ = plt.xlabel('snr riz')
_ = plt.ylabel(r'$g / \delta g$')


In [ ]:
np.isfinite(good_sub.g_gaapPsfFlux_0).sum()

In [ ]:
len(good_sub)

In [ ]:
((good_sub.g_gaapPsfFlux_0/good_sub.g_gaapPsfFluxErr_0) > 10).sum()

In [ ]:
_ = plt.hist(-2.5*np.log10(bad_sub.g_gaapPsfFlux_0/bad_sub.r_gaapPsfFlux_0), bins=np.linspace(-1, 2, 101))
_ = plt.hist(-2.5*np.log10(good_sub.g_gaapPsfFlux_0/good_sub.r_gaapPsfFlux_0), bins=np.linspace(-1, 2, 101))

In [ ]:
_ = plt.hist(bad_sub.catalog_mask_2, bins=np.linspace(-0.5, 7.5, 8), label='bad')
_ = plt.hist(good_sub.catalog_mask_2, bins=np.linspace(-0.5, 7.5, 8), label='good')
_ = plt.yscale('log')
_ = plt.legend()

In [ ]:
good_desc_bad_sub_mask = np.bitwise_and(np.where(np.isfinite(bad_sub.catalog_mask_2), bad_sub.catalog_mask_2, 0).astype(int), 7) == 7
no_object = np.bitwise_and(bad_sub.catalog_mask, 1) != 1

In [ ]:
print(np.bitwise_and(good_desc_bad_sub_mask, no_object).sum())
print(np.bitwise_and(good_desc_bad_sub_mask, ~no_object).sum())